In [ ]:
%pip install -q imagecodecs

In [ ]:
import importlib
import imagecodecs
import tifffile
import tifffile.tifffile as tifffile_core
from pathlib import Path

print("imagecodecs:", imagecodecs.__version__)
print("LZW decoder available:", hasattr(imagecodecs, "lzw_decode"))

# Refresh tifffile after installing imagecodecs
importlib.reload(tifffile_core)
tifffile = importlib.reload(tifffile)

test_file = Path(
    "/content/nightlights_frames/"
    "nightlights_flat_series_stills_india_8k.02012.tif"
)

image = tifffile.imread(test_file)
print("Decoded:", test_file.name)
print("Shape:", image.shape, "| type:", image.dtype)

In [ ]:
from pathlib import Path
import tifffile
import numpy as np
import matplotlib.pyplot as plt

FRAME_DIR = Path("/content/nightlights_frames")
paths = sorted(FRAME_DIR.glob("*.tif"))

# Check that the folder contains the expected 12 frames
print(f"Found {len(paths)} TIFF files\n")

# Read TIFF metadata without decoding all 12 images
for path in paths:
    try:
        with tifffile.TiffFile(path) as tif:
            page = tif.pages[0]
            print(
                f"{path.name} | "
                f"size: {path.stat().st_size / (1024**2):.1f} MB | "
                f"pixels: {page.shape} | "
                f"type: {page.dtype} | "
                f"compression: {page.compression.name}"
            )
    except Exception as error:
        print(f"Could not read TIFF metadata for {path.name}: {error}")

# Decode and preview just the first frame
sample_path = paths[0]
image = tifffile.imread(sample_path)
print(f"\nPreviewing: {sample_path.name}")
print(f"Array shape: {image.shape} | data type: {image.dtype}")

# Convert channel-first color images to channel-last for display
if image.ndim == 3 and image.shape[0] in (3, 4) and image.shape[-1] not in (3, 4):
    image = np.moveaxis(image, 0, -1)

# Make a small preview to keep memory use down
step = max(1, image.shape[0] // 900)
preview = image[::step, ::step]

# Scale non-8-bit data for display only
if preview.dtype != np.uint8:
    low, high = np.percentile(preview, [1, 99])
    preview = np.clip((preview.astype(np.float32) - low) / (high - low), 0, 1)

plt.figure(figsize=(10, 6))
if preview.ndim == 2:
    plt.imshow(preview, cmap="gray")
else:
    plt.imshow(preview[..., :3])
plt.title(sample_path.name)
plt.axis("off")
plt.show()

In [ ]:
from pathlib import Path
import re
import tifffile
import numpy as np
import matplotlib.pyplot as plt

FRAME_DIR = Path("/content/nightlights_frames")

def get_year(path):
    match = re.search(r"(20\d{2})", path.name)
    return int(match.group(1)) if match else None

# Find the TIFFs and sort by year
paths = sorted(
    FRAME_DIR.glob("*.tif"),
    key=lambda path: get_year(path) or 0
)

# Check image dimensions, skipping the known-bad 2014 file
valid_paths = {}
print("Frame dimensions:")

for path in paths:
    year = get_year(path)

    if year == 2014:
        print(f"{year}: skipped for now")
        continue

    try:
        with tifffile.TiffFile(path) as tif:
            shape = tif.pages[0].shape

        valid_paths[year] = path
        print(f"{year}: {shape}")

    except Exception as error:
        print(f"{year}: could not read — {error}")

if 2012 not in valid_paths or 2023 not in valid_paths:
    raise ValueError("Could not find readable frames for both 2012 and 2023.")

# Read a reduced version of each endpoint frame for a quick visual check
def read_small_rgb(path, step=8):
    image = tifffile.imread(path)
    image = image[::step, ::step, :3]
    return image.astype(np.float32)

image_2012 = read_small_rgb(valid_paths[2012])
image_2023 = read_small_rgb(valid_paths[2023])

# Use one shared display scale so the two previews can be compared fairly
combined = np.concatenate(
    [image_2012.reshape(-1), image_2023.reshape(-1)]
)
low, high = np.percentile(combined, [1, 99.9])

def display_scale(image):
    return np.clip((image - low) / (high - low), 0, 1)

preview_2012 = display_scale(image_2012)
preview_2023 = display_scale(image_2023)
overlay = 0.5 * preview_2012 + 0.5 * preview_2023

# The shared geography should overlap in the blended panel
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(preview_2012)
axes[0].set_title("2012")

axes[1].imshow(preview_2023)
axes[1].set_title("2023")

axes[2].imshow(overlay)
axes[2].set_title("2012 and 2023 blended")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import tifffile
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

frame_2012 = Path(
    "/content/nightlights_frames/"
    "nightlights_flat_series_stills_india_8k.02012.tif"
)

# Read the frame and drop the alpha channel for display
image = tifffile.imread(frame_2012)
rgb = image[..., :3]

# Downsample the preview for faster display.
# Coordinates on the axes still refer to the original 7680 × 4320 image.
step = 8
preview = rgb[::step, ::step].astype(np.float32)

# Scale this preview for viewing only
low, high = np.percentile(preview, [1, 99.9])
preview = np.clip((preview - low) / (high - low), 0, 1)

height, width = image.shape[:2]

fig, ax = plt.subplots(figsize=(16, 9))
ax.imshow(
    preview,
    extent=[0, width, height, 0],
    interpolation="nearest"
)

# Fine grid lines every 500 pixels; labels every 1000 pixels
ax.set_xticks(np.arange(0, width + 1, 1000))
ax.set_yticks(np.arange(0, height + 1, 1000))
ax.set_xticks(np.arange(0, width + 1, 500), minor=True)
ax.set_yticks(np.arange(0, height + 1, 500), minor=True)
ax.grid(which="minor", color="white", alpha=0.35, linewidth=0.6)

ax.set_xlabel("Pixel column from left")
ax.set_ylabel("Pixel row from top")
ax.set_title("2012 frame: reference grid for the India crop")
plt.show()

In [ ]:
from pathlib import Path
import tifffile
import numpy as np
import matplotlib.pyplot as plt

frame_2012 = Path(
    "/content/nightlights_frames/"
    "nightlights_flat_series_stills_india_8k.02012.tif"
)

image = tifffile.imread(frame_2012)

# Provisional crop bounds in original image pixels
x0, x1 = 900, 6000
y0, y1 = 900, 4200

# Keep RGB for display and later visual analysis
crop = image[y0:y1, x0:x1, :3]

# Downsample only the preview; the crop itself stays full resolution
step = 4
preview = crop[::step, ::step].astype(np.float32)

# Display scaling only
low, high = np.percentile(preview, [1, 99.9])
preview = np.clip((preview - low) / (high - low), 0, 1)

fig, ax = plt.subplots(figsize=(12, 9))
ax.imshow(
    preview,
    extent=[x0, x1, y1, y0],
    interpolation="nearest"
)

ax.set_xticks(np.arange(x0, x1 + 1, 500))
ax.set_yticks(np.arange(y0, y1 + 1, 500))
ax.grid(color="white", alpha=0.35, linewidth=0.6)

ax.set_xlabel("Original image column")
ax.set_ylabel("Original image row")
ax.set_title("Provisional India-region crop from 2012")
plt.show()

In [ ]:
from pathlib import Path
import tifffile

frame_2012 = Path(
    "/content/nightlights_frames/"
    "nightlights_flat_series_stills_india_8k.02012.tif"
)

# These TIFF tags can store geographic coordinates and map projection details
geo_tags = {
    33550: "ModelPixelScaleTag",
    33922: "ModelTiepointTag",
    34264: "ModelTransformationTag",
    34735: "GeoKeyDirectoryTag",
}

with tifffile.TiffFile(frame_2012) as tif:
    page = tif.pages[0]
    print("Image shape:", page.shape)

    found_geo_tags = False

    for tag_id, tag_name in geo_tags.items():
        tag = page.tags.get(tag_id)

        if tag is not None:
            found_geo_tags = True
            print(f"{tag_name}: {tag.value}")

    if not found_geo_tags:
        print("No standard GeoTIFF coordinate tags found.")

In [ ]:
from pathlib import Path
import tifffile
import numpy as np
import matplotlib.pyplot as plt

frame_2012 = Path(
    "/content/nightlights_frames/"
    "nightlights_flat_series_stills_india_8k.02012.tif"
)

# Shared crop bounds, in original image pixels
x0, x1 = 900, 6000
y0, y1 = 900, 4200

image = tifffile.imread(frame_2012)
rgb = image[y0:y1, x0:x1, :3].astype(np.float32)

# Convert RGB to a single brightness value.
# Normalize by the uint16 maximum so values are on a 0–1 scale.
dtype_max = np.iinfo(image.dtype).max
rgb /= dtype_max
brightness = (
    0.2126 * rgb[..., 0]
    + 0.7152 * rgb[..., 1]
    + 0.0722 * rgb[..., 2]
)

# Exclude the year stamp, using its approximate position in the original frame
crop_x = np.arange(x0, x1)[None, :]
crop_y = np.arange(y0, y1)[:, None]
year_stamp = (
    (crop_x >= 2450) & (crop_x <= 2850)
    & (crop_y >= 2750) & (crop_y <= 3030)
)
brightness[year_stamp] = np.nan

# Report brightness percentiles, excluding the year stamp
values = brightness[np.isfinite(brightness)]
percentiles = [50, 75, 90, 95, 99, 99.5, 99.9, 100]

print("2012 crop brightness percentiles (normalized 0–1):")
for percentile, value in zip(
    percentiles,
    np.percentile(values, percentiles)
):
    print(f"{percentile:>5}%: {value:.4f}")

# Preview RGB, brightness, and the distribution
step = 4
rgb_preview = rgb[::step, ::step]
brightness_preview = brightness[::step, ::step]

display_scale = np.percentile(values, 99.9)
rgb_display = np.clip(rgb_preview / display_scale, 0, 1)

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

axes[0].imshow(rgb_display)
axes[0].set_title("2012 crop (display-stretched preview)")

axes[1].imshow(
    brightness_preview,
    cmap="inferno",
    vmin=0,
    vmax=display_scale
)
axes[1].set_title("Brightness preview")

axes[2].hist(values[::20], bins=100, range=(0, 1), log=True)
axes[2].set_title("Brightness distribution")
axes[2].set_xlabel("Normalized brightness")
axes[2].set_ylabel("Sampled pixel count")

for ax in axes[:2]:
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
from pathlib import Path
import re
import tifffile
import numpy as np
import matplotlib.pyplot as plt

FRAME_DIR = Path("/content/nightlights_frames")

def get_year(path):
    match = re.search(r"(20\d{2})", path.name)
    return int(match.group(1)) if match else None

paths_by_year = {
    get_year(path): path
    for path in FRAME_DIR.glob("*.tif")
    if get_year(path) is not None
}

years = [2012, 2023]
thresholds = [0.15, 0.20, 0.25]

# Keep these bounds identical for every frame
x0, x1 = 900, 6000
y0, y1 = 900, 4200

# Exclude the year stamp from both the display and pixel counts
label_x0, label_x1 = 2450, 2850
label_y0, label_y1 = 2750, 3030

def inspect_year(year):
    image = tifffile.imread(paths_by_year[year])
    rgb = image[y0:y1, x0:x1, :3].astype(np.float32) / 65535.0

    height, width = rgb.shape[:2]
    original_x = np.arange(x0, x1)[None, :]
    original_y = np.arange(y0, y1)[:, None]

    year_stamp = (
        (original_x >= label_x0) & (original_x <= label_x1)
        & (original_y >= label_y0) & (original_y <= label_y1)
    )
    valid_pixels = ~year_stamp

    brightness = (
        0.2126 * rgb[..., 0]
        + 0.7152 * rgb[..., 1]
        + 0.0722 * rgb[..., 2]
    )

    # Save a smaller preview to keep memory use down
    step = 6
    rgb_preview = rgb[::step, ::step].copy()
    label_preview = year_stamp[::step, ::step]
    rgb_preview[label_preview] = 0

    # Display stretch only; threshold calculations use the original 0–1 values
    rgb_preview = np.clip(rgb_preview / 0.924, 0, 1)

    mask_previews = {}
    coverages = {}

    for threshold in thresholds:
        mask = (brightness >= threshold) & valid_pixels
        mask_previews[threshold] = mask[::step, ::step]
        coverages[threshold] = 100 * mask.sum() / valid_pixels.sum()

    return rgb_preview, mask_previews, coverages

results = {year: inspect_year(year) for year in years}

fig, axes = plt.subplots(2, 4, figsize=(18, 9))

for row, year in enumerate(years):
    rgb_preview, mask_previews, coverages = results[year]

    axes[row, 0].imshow(rgb_preview)
    axes[row, 0].set_title(f"{year}: crop preview")
    axes[row, 0].axis("off")

    for col, threshold in enumerate(thresholds, start=1):
        axes[row, col].imshow(mask_previews[threshold], cmap="gray", vmin=0, vmax=1)
        axes[row, col].set_title(
            f"Threshold {threshold:.2f}\n"
            f"Lit pixels: {coverages[threshold]:.2f}%"
        )
        axes[row, col].axis("off")

plt.suptitle("Compare candidate light thresholds using one shared scale")
plt.tight_layout()
plt.show()

print("Share of crop pixels detected as lit:")
for year in years:
    coverages = results[year][2]
    print(
        f"{year}: "
        + ", ".join(
            f"{threshold:.2f} → {coverages[threshold]:.2f}%"
            for threshold in thresholds
        )
    )

In [ ]:
from pathlib import Path
import re
import tifffile
import numpy as np
import matplotlib.pyplot as plt

FRAME_DIR = Path("/content/nightlights_frames")

# Shared crop bounds
x0, x1 = 900, 6000
y0, y1 = 900, 4200

# Provisional light threshold selected from the previews
LIGHT_THRESHOLD = 0.20

# Grid shape: 16 columns scanned west to east, 10 rows north to south
GRID_ROWS = 10
GRID_COLS = 16

def get_year(path):
    match = re.search(r"(20\d{2})", path.name)
    return int(match.group(1)) if match else None

coverage_grids = {}
brightness_grids = {}

for path in sorted(FRAME_DIR.glob("*.tif")):
    year = get_year(path)

    # if year == 2014:
    #     print("Skipping 2014 for now.")
    #     continue

    try:
        image = tifffile.imread(path)
    except Exception as error:
        print(f"Skipping {year}: could not decode TIFF — {error}")
        continue

    # Convert the cropped RGB values to a 0–1 range
    dtype_max = np.iinfo(image.dtype).max
    rgb = image[y0:y1, x0:x1, :3].astype(np.float32) / dtype_max

    # One brightness value per pixel
    brightness = (
        0.2126 * rgb[..., 0]
        + 0.7152 * rgb[..., 1]
        + 0.0722 * rgb[..., 2]
    )

    crop_height, crop_width = brightness.shape
    original_x = np.arange(x0, x1)[None, :]
    original_y = np.arange(y0, y1)[:, None]

    # Exclude the year stamp so it cannot become a false bright spot
    year_stamp = (
        (original_x >= 2450) & (original_x <= 2850)
        & (original_y >= 2750) & (original_y <= 3030)
    )
    valid_pixels = ~year_stamp

    coverage = np.zeros((GRID_ROWS, GRID_COLS), dtype=np.float32)
    mean_light_brightness = np.zeros_like(coverage)

    row_edges = np.linspace(0, crop_height, GRID_ROWS + 1, dtype=int)
    col_edges = np.linspace(0, crop_width, GRID_COLS + 1, dtype=int)

    for row in range(GRID_ROWS):
        for col in range(GRID_COLS):
            r0, r1 = row_edges[row], row_edges[row + 1]
            c0, c1 = col_edges[col], col_edges[col + 1]

            cell_brightness = brightness[r0:r1, c0:c1]
            cell_valid = valid_pixels[r0:r1, c0:c1]
            lit = cell_valid & (cell_brightness >= LIGHT_THRESHOLD)

            if cell_valid.any():
                coverage[row, col] = lit.sum() / cell_valid.sum()

            if lit.any():
                mean_light_brightness[row, col] = cell_brightness[lit].mean()

    coverage_grids[year] = coverage
    brightness_grids[year] = mean_light_brightness

    print(
        f"{year}: detected-lit pixels "
        f"{100 * coverage.mean():.2f}% of the crop"
    )

# Compare the first and last available years on shared color scales
if 2012 not in coverage_grids or 2023 not in coverage_grids:
    raise ValueError("Could not create both the 2012 and 2023 grids.")

coverage_max = max(grid.max() for grid in coverage_grids.values())

fig, axes = plt.subplots(2, 2, figsize=(14, 9))

for col, year in enumerate([2012, 2023]):
    coverage_image = axes[0, col].imshow(
        coverage_grids[year],
        origin="upper",
        cmap="magma",
        vmin=0,
        vmax=coverage_max,
        aspect="auto"
    )
    axes[0, col].set_title(f"{year}: lit-pixel coverage")
    axes[0, col].set_xlabel("West → East column")
    axes[0, col].set_ylabel("North → South row")
    fig.colorbar(coverage_image, ax=axes[0, col], label="Fraction lit")

    brightness_image = axes[1, col].imshow(
        brightness_grids[year],
        origin="upper",
        cmap="viridis",
        vmin=LIGHT_THRESHOLD,
        vmax=1,
        aspect="auto"
    )
    axes[1, col].set_title(f"{year}: mean brightness of detected lights")
    axes[1, col].set_xlabel("West → East column")
    axes[1, col].set_ylabel("North → South row")
    fig.colorbar(brightness_image, ax=axes[1, col], label="Brightness (0–1)")

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
from IPython.display import Audio, display

# A 10-note pentatonic scale: south/bottom is low, north/top is high
bottom_to_top = np.array([
    261.63, 293.66, 329.63, 392.00, 440.00,
    523.25, 587.33, 659.25, 783.99, 880.00
])
row_frequencies = bottom_to_top[::-1]  # Grid row 0 is north/top

sample_rate = 22050
column_seconds = 0.4
samples_per_column = int(sample_rate * column_seconds)
grid_rows, grid_cols = coverage_grids[2012].shape

# Use common caps from all available years.
# This lets the same coverage and brightness produce the same sound in each year.
all_coverage = np.concatenate([
    grid[grid > 0] for grid in coverage_grids.values()
])
all_brightness = np.concatenate([
    grid[grid > 0] for grid in brightness_grids.values()
])

coverage_cap = np.percentile(all_coverage, 95)
brightness_cap = np.percentile(all_brightness, 95)

# Soft fades prevent clicks between scanned columns
t = np.arange(samples_per_column) / sample_rate
fade_n = int(0.012 * sample_rate)
envelope = np.ones(samples_per_column)
envelope[:fade_n] = np.linspace(0, 1, fade_n)
envelope[-fade_n:] = np.linspace(1, 0, fade_n)

audio_tracks = {}

for year in sorted(coverage_grids):
    coverage = coverage_grids[year]
    brightness = brightness_grids[year]
    track = np.zeros(grid_cols * samples_per_column, dtype=np.float32)

    for col in range(grid_cols):
        chord = np.zeros(samples_per_column, dtype=np.float32)
        active_notes = 0

        # Each cell in this column contributes a note.
        # All notes in a column play together as a chord.
        for row in range(grid_rows):
            cell_coverage = coverage[row, col]

            # No detected light means this cell is silent
            if cell_coverage <= 0:
                continue

            # More lit pixels = louder note, using one scale for all years
            volume_level = np.clip(cell_coverage / coverage_cap, 0, 1)
            amplitude = 0.12 * np.sqrt(volume_level)

            # Brighter detected pixels add a little second harmonic,
            # making the synthesized tone sound brighter.
            cell_brightness = brightness[row, col]
            color_level = np.clip(
                cell_brightness / brightness_cap, 0, 1
            )

            frequency = row_frequencies[row]
            phase = 2 * np.pi * frequency * t

            tone = (
                (1 - 0.22 * color_level) * np.sin(phase)
                + (0.22 * color_level) * np.sin(2 * phase)
            )

            chord += amplitude * tone * envelope
            active_notes += 1

        # Keep a large chord from becoming much louder just because
        # several rows in the column contain lights.
        if active_notes:
            chord /= np.sqrt(active_notes)

        start = col * samples_per_column
        end = start + samples_per_column
        track[start:end] = chord

    audio_tracks[year] = track

# Apply one safety gain to the whole set, not a separate gain per year
peak = max(np.max(np.abs(track)) for track in audio_tracks.values())
shared_gain = min(0.8 / peak, 1.0) if peak > 0 else 1.0

for year in audio_tracks:
    audio_tracks[year] *= shared_gain

print(f"Each year’s scan lasts {grid_cols * column_seconds:.1f} seconds.")
print(f"Years available: {', '.join(map(str, sorted(audio_tracks)))}")
print(f"2012 RMS level: {np.sqrt(np.mean(audio_tracks[2012] ** 2)):.4f}")
print(f"2023 RMS level: {np.sqrt(np.mean(audio_tracks[2023] ** 2)):.4f}")

print("2012 preview")
display(Audio(audio_tracks[2012], rate=sample_rate, normalize=False))

print("2023 preview")
display(Audio(audio_tracks[2023], rate=sample_rate, normalize=False))

In [ ]:
from pathlib import Path
import re
import shutil
import subprocess

import numpy as np
import tifffile
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from scipy.ndimage import gaussian_filter1d
from scipy.io.wavfile import write as write_wav

FRAME_DIR = Path("/content/nightlights_frames")
OUTPUT_DIR = Path("/content/nightlights_year_videos")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Same crop used to create the audio grids
x0, x1 = 900, 6000
y0, y1 = 900, 4200

# Keep these settings matched to the audio-generation cell
column_seconds = 0.4
sample_rate = 22050
fps = 60
display_scale = 0.924  # Same display scale for every year

if shutil.which("ffmpeg") is None:
    raise RuntimeError("FFmpeg is unavailable in this Colab runtime.")


def get_year(path):
    match = re.search(r"(20\d{2})", path.name)
    return int(match.group(1)) if match else None


frame_paths = {
    get_year(path): path
    for path in FRAME_DIR.glob("*.tif")
    if get_year(path) is not None
}


def to_int16(audio):
    return (np.clip(audio, -1, 1) * 32767).astype(np.int16)


for year in sorted(audio_tracks):
    if year not in frame_paths:
        print(f"Skipping {year}: no matching frame found.")
        continue

    # Load this year's map and crop it to the shared region
    image = tifffile.imread(frame_paths[year])
    dtype_max = np.iinfo(image.dtype).max
    rgb = image[y0:y1, x0:x1, :3].astype(np.float32) / dtype_max

    # Downsample the displayed map for faster video rendering.
    # The crop bounds and sound grid are unchanged.
    preview = np.clip(rgb[::4, ::4] / display_scale, 0, 1)

    duration = len(audio_tracks[year]) / sample_rate
    total_frames = max(2, round(duration * fps))
    grid_cols = coverage_grids[year].shape[1]

    # Make a smooth, non-repeating variation in the bar's speed.
    # The bar still starts and ends at the same positions.
    rng = np.random.default_rng(year)
    speed_variation = gaussian_filter1d(
        rng.normal(0, 1, total_frames - 1),
        sigma=max(1, total_frames / 30)
    )
    speed = np.clip(1 + 0.12 * speed_variation, 0.65, 1.35)

    scan_progress = np.concatenate(([0], np.cumsum(speed)))
    scan_progress /= scan_progress[-1]

    fig, ax = plt.subplots(figsize=(10, 6.5))

    ax.imshow(
        preview,
        extent=[x0, x1, y1, y0],
        origin="upper",
        interpolation="nearest",
        aspect="auto"
    )

    # Show the 16 sound columns faintly on the map
    for col in range(1, grid_cols):
        boundary_x = x0 + col * (x1 - x0) / grid_cols
        ax.axvline(
            boundary_x,
            color="white",
            linewidth=0.6,
            alpha=0.25
        )

    ax.set_xlim(x0, x1)
    ax.set_ylim(y1, y0)
    ax.set_title(f"Night lights around India — {year}")
    ax.set_xlabel("Scan direction: west → east")
    ax.set_ylabel("North ↑")
    ax.grid(False)

    # Soft grey bar with a subdued outline
    bar_outline = ax.axvline(
        x0,
        color="black",
        linewidth=4,
        alpha=0.45
    )
    bar = ax.axvline(
        x0,
        color="#E5E7EB",
        linewidth=2.5,
        alpha=0.85
    )

    status = ax.text(
        0.02,
        0.97,
        "",
        transform=ax.transAxes,
        va="top",
        color="white",
        bbox=dict(
            facecolor="black",
            alpha=0.65,
            edgecolor="none"
        )
    )

    def update(frame):
        elapsed = frame / fps
        progress = scan_progress[frame]

        # Bar travels across the crop over the same duration as the audio
        x = x0 + progress * (x1 - x0)
        column_number = min(
            int(elapsed / column_seconds) + 1,
            grid_cols
        )

        bar_outline.set_xdata([x, x])
        bar.set_xdata([x, x])
        status.set_text(
            f"{year}  |  Column {column_number} of {grid_cols}"
        )

        return bar_outline, bar, status

    animation = FuncAnimation(
        fig,
        update,
        frames=total_frames,
        interval=1000 / fps,
        blit=False
    )

    wav_path = OUTPUT_DIR / f"nightlights_{year}.wav"
    silent_path = OUTPUT_DIR / f"nightlights_{year}_silent.mp4"
    final_path = OUTPUT_DIR / f"nightlights_{year}_with_scan.mp4"

    # Save this year's audio as its own WAV
    write_wav(
        wav_path,
        sample_rate,
        to_int16(audio_tracks[year])
    )

    # Render the moving bar, then combine it with the matching year's audio
    animation.save(
        silent_path,
        writer="ffmpeg",
        fps=fps,
        dpi=100
    )
    plt.close(fig)

    subprocess.run(
        [
            "ffmpeg",
            "-y",
            "-i", str(silent_path),
            "-i", str(wav_path),
            "-map", "0:v:0",
            "-map", "1:a:0",
            "-c:v", "libx264",
            "-pix_fmt", "yuv420p",
            "-c:a", "aac",
            "-shortest",
            str(final_path)
        ],
        check=True
    )

    silent_path.unlink(missing_ok=True)
    print(f"Saved {year}: {final_path.name}")